# triangle-splatting  2 :: Custom data :: duke_statue

-----
- Conda env : [triangle_splatting2](README.md#setup-a-conda-environment)
-----

### Check system

In [17]:
!nvidia-smi

Wed Oct 15 15:28:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 22%   42C    P8             31W /  250W |     533MiB /  11264MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Download a video

In [18]:
import os
from pathlib import Path

Path("./temp_data").mkdir(exist_ok=True, parents=True)

In [19]:
import gdown

VIDEO_NAME = "duke_statue"

FPS = 10
RES = 4
id = "1fFvegLaEc_pGmputDUD6InIQFf8auEC1"
vid_path = f"./temp_data/{VIDEO_NAME}.mov"

if not os.path.exists(vid_path):
    gdown.download(id=id, output = vid_path)
else:
    print(f"{vid_path} already is downloaded")

./temp_data/duke_statue.mov already is downloaded


### Extract images from the video

In [20]:
DATASET_DIR_PATH = f"./temp_data/{VIDEO_NAME}"
IMAGES_DIR_PATH = os.path.join(DATASET_DIR_PATH, "images")
DATABASE_PATH = os.path.join(DATASET_DIR_PATH, "database.db")

if not os.path.exists(IMAGES_DIR_PATH):
    Path(IMAGES_DIR_PATH).mkdir(exist_ok=True, parents=True)
    !ffmpeg -i $vid_path -vf fps=$FPS $IMAGES_DIR_PATH/frame_%04d.jpg
else:
    print(f"{IMAGES_DIR_PATH} already is extracted")

./temp_data/duke_statue/images already is extracted


### Colmap :: Feature Extraction & Matching & Reconstruction

In [21]:
SPARSE_DIR = os.path.join(DATASET_DIR_PATH, "sparse")
Path(SPARSE_DIR).mkdir(exist_ok=True, parents=True)

if not os.path.exists(DATABASE_PATH):
    # Colmap Feature Extraction
    !colmap feature_extractor \
        --database_path $DATABASE_PATH \
        --image_path $IMAGES_DIR_PATH  --ImageReader.camera_model PINHOLE
    !colmap sequential_matcher \
        --database_path $DATABASE_PATH
    !colmap mapper \
        --database_path $DATABASE_PATH \
        --image_path $IMAGES_DIR_PATH \
        --output_path $SPARSE_DIR
else:
    print(f"{DATABASE_PATH} already is extracted and reconstructed")

./temp_data/duke_statue/database.db already is extracted and reconstructed


### Triangle-Splatting :: Training (Outdoor mode)

In [8]:
DATASET_DIR_PATH
OUTDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_outdoor"
print(DATASET_DIR_PATH)
print(OUTDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/train.py -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --eval


./temp_data/duke_statue
./temp_result/duke_statue_outdoor
Optimizing ./temp_result/duke_statue_outdoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output f

### Triangle-Splatting :: Rendering (Outdoor mode)

In [9]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/render.py -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES -s $DATASET_DIR_PATH

Looking for config file in ./temp_result/duke_statue_outdoor/cfg_args
Config file found: ./temp_result/duke_statue_outdoor/cfg_args
Rendering ./temp_result/duke_statue_outdoor
Loading trained model at iteration 30000 [15/10 07:16:45]
Using cache found in /home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbone

### Triangle-Splatting :: Create a video (Outdoor mode)

In [10]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_video.py  -s $DATASET_DIR_PATH -m $OUTDOOR_OUTPUR_DIR_PATH -r $RES --save_as $OUTDOOR_OUTPUR_DIR_PATH/output_video

Looking for config file in ./temp_result/duke_statue_outdoor/cfg_args
Config file found: ./temp_result/duke_statue_outdoor/cfg_args
Creating video for ./temp_result/duke_statue_outdoor
Loading trained model at iteration 30000
Using cache found in /home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbones/ViT_DI

In [11]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_ply.py $OUTDOOR_OUTPUR_DIR_PATH/point_cloud/iteration_30000 --out $OUTDOOR_OUTPUR_DIR_PATH/mesh.ply

Saved PLY to: ./temp_result/duke_statue_outdoor/mesh.ply
Vertices: 124916, Faces: 61066


### Triangle-Splatting :: Training (Indoor mode)

In [12]:
DATASET_DIR_PATH
INDOOR_OUTPUR_DIR_PATH = f"./temp_result/{VIDEO_NAME}_indoor"
print(DATASET_DIR_PATH)
print(INDOOR_OUTPUR_DIR_PATH)

!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/train.py -s $DATASET_DIR_PATH -m $INDOOR_OUTPUR_DIR_PATH -r $RES --eval  --indoor 

./temp_data/duke_statue
./temp_result/duke_statue_indoor
Optimizing ./temp_result/duke_statue_indoor
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/lpips/weights/v0.1/vgg.pth
Output fol

### Triangle-Splatting :: Rendering (Indoor mode)

In [13]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/render.py -m $INDOOR_OUTPUR_DIR_PATH

Looking for config file in ./temp_result/duke_statue_indoor/cfg_args
Config file found: ./temp_result/duke_statue_indoor/cfg_args
Rendering ./temp_result/duke_statue_indoor
Loading trained model at iteration 30000 [15/10 07:29:45]
Using cache found in /home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbones/V

### Triangle-Splatting :: Create a video (Indoor mode)

In [14]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_video.py -s $DATASET_DIR_PATH -m $INDOOR_OUTPUR_DIR_PATH -r $RES --save_as $INDOOR_OUTPUR_DIR_PATH/output_video

Looking for config file in ./temp_result/duke_statue_indoor/cfg_args
Config file found: ./temp_result/duke_statue_indoor/cfg_args
Creating video for ./temp_result/duke_statue_indoor
Loading trained model at iteration 30000
Using cache found in /home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/hyunjae/anaconda3/envs/triangle_splatting2/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/hyunjae/.cache/torch/hub/yvanyin_metric3d_main/mono/model/backbones/ViT_DINO.

In [16]:
!CUDA_VISIBLE_DEVICES=0 python temp_triangle-splatting2/create_ply.py $INDOOR_OUTPUR_DIR_PATH/point_cloud/iteration_30000 --out $INDOOR_OUTPUR_DIR_PATH/mesh.ply

Saved PLY to: ./temp_result/duke_statue_indoor/mesh.ply
Vertices: 96997, Faces: 58088
